# 线性方程组与向量空间

学习目标：用消元和向量空间解释线性方程组的解集，判断有解与唯一性，并用秩和残差核对小型计算。

前置知识：线性组合、矩阵乘法、向量与矩阵形状；Python 数组索引、循环与模块导入。

运行环境：Python 3.12、NumPy 2.5；本章讨论实数系数与实数解。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 把方程组写成矩阵

两个未知数满足 $x_1+x_2=3$、$2x_1-x_2=0$。解必须同时满足两条方程，不能只符合其中一条。把系数按行排列，得到

$$Ax=b,\qquad A=\begin{bmatrix}1&1\\2&-1\end{bmatrix},\quad
x=\begin{bmatrix}x_1\\x_2\end{bmatrix},\quad b=\begin{bmatrix}3\\0\end{bmatrix}.$$

一般地，$A\in\mathbb{R}^{m\times n}$ 是系数矩阵，$x\in\mathbb{R}^n$ 是未知向量，$b\in\mathbb{R}^m$ 是已知右侧；$m,n$ 分别为方程数与未知数数目，都是正整数。这里的线性是指未知数只按一次项相加，不出现未知数间的乘积或平方。

所有满足方程的 x 构成解集。给定候选解 $\hat{x}$，残差 $r=A\hat{x}-b\in\mathbb{R}^m$ 逐项反映它是否满足原方程；精确解的残差为零。本章数学向量按列写，代码用一维数组保存单个向量。

In [1]:
import numpy as np

a = np.array([[1.0, 1.0], [2.0, -1.0]])
b = np.array([3.0, 0.0])
candidate = np.array([1.0, 2.0])

print(a.shape, candidate.shape, b.shape)  # (2, 2) (2,) (2,)
print(a @ candidate - b)  # 预期：[0. 0.]，同时满足两个方程。
print(a @ np.array([3.0, 0.0]) - b)  # 预期：[0. 6.]，只满足第一条。

(2, 2) (2,) (2,)
[0. 0.]
[0. 6.]


## 2 消元与自由变量

### 2.1 用等价方程消去未知数

增广矩阵 $[A\mid b]$ 把 b 接在 A 的右侧。以下行操作保持解集：交换两行；整行乘非零数；把一行的倍数加到另一行。对增广矩阵操作时，右侧常数必须一起变化。

记 $R_i$ 为第 i 行，箭头表示“把该行替换为右侧结果”。对上例做 $R_2\leftarrow R_2-2R_1$：

$$\left[\begin{array}{cc|c}1&1&3\\2&-1&0\end{array}\right]
\longrightarrow
\left[\begin{array}{cc|c}1&1&3\\0&-3&-6\end{array}\right].$$

第二行给出 $x_2=2$，代回第一行得 $x_1=1$，这叫回代。代码只复现这两行的手算，不编写通用消元求解器。

In [2]:
augmented = np.column_stack((a, b))  # 把 b 接为最后一列。
reduced = augmented.copy()
reduced[1] = reduced[1] - 2 * reduced[0]
print(reduced)  # 预期：[[1, 1, 3], [0, -3, -6]]。

x2 = reduced[1, 2] / reduced[1, 1]
x1 = (reduced[0, 2] - reduced[0, 1] * x2) / reduced[0, 0]
manual_solution = np.array([x1, x2])
print(manual_solution)  # 预期：[1. 2.]。
print(a @ manual_solution - b)  # 预期：[0. 0.]，代回原方程。

[[ 1.  1.  3.]
 [ 0. -3. -6.]]
[1. 2.]
[0. 0.]


### 2.2 主元与阶梯形

阶梯形矩阵的零行在底部，每个非零行的首个非零元素位于上一行首个非零元素的右方，其下方元素为零。这些首个非零元素称为主元（pivot）；包含主元的位置与列分别称为主元位置、主元列。

继续把主元变成 1，并清除它上方的元素，就得到简化行阶梯形。上例再做 $R_2\leftarrow-R_2/3$、$R_1\leftarrow R_1-R_2$，可直接读出两个未知数。

消元时若当前待选元素是 0，应寻找下方非零行交换；不能除以 0。这里复现的是精确小例子的代数步骤，浮点求解还需要考虑数值误差。

In [3]:
reduced[1] = reduced[1] / -3
reduced[0] = reduced[0] - reduced[1]
print(reduced)  # 预期：[[1, 0, 1], [0, 1, 2]]。
print(reduced[:, -1])  # 预期：[1. 2.]；本例每个未知数列都有主元。

[[ 1.  0.  1.]
 [-0.  1.  2.]]
[1. 2.]


### 2.3 无解与无穷多解

若方程为 $x_1+2x_2=3$、$2x_1+4x_2=6$，消去第二行后得到 $0=0$。它没有增加约束。取自由变量 $x_2=t$，其中 $t\in\mathbb{R}$ 可任取，全部解为

$$x=\begin{bmatrix}3-2t\\t\end{bmatrix}.
$$

如果第二个常数改成 7，同样的消元会得到 $0=1$，说明没有解。方程组有解时称为相容（consistent），无解时称为不相容（inconsistent）。

在实数范围，线性方程组只有三种解的数量：无解、唯一解或无穷多解。相容系统中，没有主元的未知数列对应自由变量；有自由变量便有无穷多个解。

In [4]:
dependent = np.array([[1.0, 2.0], [2.0, 4.0]])
for right_side in (np.array([3.0, 6.0]), np.array([3.0, 7.0])):
    work = np.column_stack((dependent, right_side))
    work[1] = work[1] - 2 * work[0]
    print(work[1])  # 依次为 [0, 0, 0] 与 [0, 0, 1]。

for parameter in (-1.0, 0.0, 2.0):
    solution = np.array([3 - 2 * parameter, parameter])
    print(solution, dependent @ solution - np.array([3.0, 6.0]))  # 三组解为 [5, -1]、[3, 0]、[-1, 2]；每组方程残差都为 [0, 0]。
# 三个不同解的残差均为零；全部解仍由参数公式描述。

[0. 0. 0.]
[0. 0. 1.]
[ 5. -1.] [0. 0.]
[3. 0.] [0. 0.]
[-1.  2.] [0. 0.]


## 3 张成与线性无关

### 3.1 能组合出哪些向量

给定 $v_1,\ldots,v_k\in\mathbb{R}^m$，其中 k 为正整数，它们的张成（span）是全部线性组合组成的集合：

$$\operatorname{span}(v_1,\ldots,v_k)
=\left\{\sum_{j=1}^{k}c_jv_j:c_1,\ldots,c_k\in\mathbb{R}\right\}.$$

大括号内的冒号读作“其中”。这不是只挑几组系数得到的样本，而是允许所有实数系数的结果。

取 $v_1=\begin{bmatrix}1\\1\\0\end{bmatrix}$、$v_2=\begin{bmatrix}1\\0\\1\end{bmatrix}$，组合 $sv_1+tv_2=\begin{bmatrix}s+t\\s\\t\end{bmatrix}$，其中 $s,t\in\mathbb{R}$。所以结果的第一个分量总等于后两个分量之和；反过来，满足这一关系的向量都能这样表示。

In [5]:
v1 = np.array([1.0, 1.0, 0.0])
v2 = np.array([1.0, 0.0, 1.0])
target = 1 * v1 + 3 * v2

print(target)  # 预期：[4. 1. 3.]，属于这两个向量的张成。
print(target[0], target[1] + target[2])  # 均为 4.0。
outside = np.array([4.0, 1.0, 4.0])
print(outside[0] - outside[1] - outside[2])  # 预期：-1.0，不满足所需关系。

[4. 1. 3.]
4.0 4.0
-1.0


### 3.2 是否有多余的向量

若 $c_1v_1+\cdots+c_kv_k=0$ 只有 $c_1=\cdots=c_k=0$ 这一组解，称这组向量线性无关（linearly independent）；若存在不全为零的系数使等式成立，则线性相关。

对上例，$sv_1+tv_2=0$ 的第二、三个分量分别要求 $s=0$、$t=0$，所以二者无关。若再加入 $v_3=v_1+v_2$，便有 $v_1+v_2-v_3=0$；第三个向量没有扩大张成。

这是整组向量的性质。只检查每两列不成倍数还不够：本例三列两两不成倍数，整组仍相关。含零向量的向量组也必定相关。

In [6]:
v3 = v1 + v2
columns = np.column_stack((v1, v2, v3))
relation = np.array([1.0, 1.0, -1.0])

print(columns)  # 三列分别为 v1、v2、v3。
print(columns @ relation)  # 预期：[0. 0. 0.]，系数并不全为零。
print(2 * v1 + 4 * v2, 1 * v1 + 3 * v2 + v3)  # 两种系数组合都得到 [6, 2, 4]。
# 两种系数组合同得 [6. 2. 4.]；相关列让表示不唯一。

[[1. 1. 2.]
 [1. 0. 1.]
 [0. 1. 1.]]
[0. 0. 0.]
[6. 2. 4.] [6. 2. 4.]


## 4 向量空间、列空间与零空间

### 4.1 子空间对加法和数乘封闭

本章研究的向量空间是实坐标空间 $\mathbb{R}^n$ 及其线性子空间，运算沿用逐分量加法和实数数乘。

子空间 $V\subseteq\mathbb{R}^n$ 要包含零向量，并对加法和数乘封闭：若 $u,v\in V$、$c\in\mathbb{R}$，则 $u+v\in V$、$cu\in V$。“封闭”表示操作后仍在集合内。

上节的张成就是子空间。例如 $V=\{(s+t,s,t)^{\mathsf T}:s,t\in\mathbb{R}\}$：零系数给出零向量，组合相加或数乘后仍有相同形式。相比之下，$W=\{(1,t)^{\mathsf T}:t\in\mathbb{R}\}$ 不包含零向量，不是子空间。

In [7]:
first = 1 * v1 + 3 * v2
second = -2 * v1 + 1 * v2
combined = first + second
scaled = -2 * first

print(combined[0] - combined[1] - combined[2])  # 预期：0.0。
print(scaled[0] - scaled[1] - scaled[2])  # 预期：0.0。
point_on_w = np.array([1.0, 2.0])
print(0 * point_on_w)  # [0. 0.] 不再满足 W 的第一坐标为 1，数乘不封闭。

0.0
0.0
[0. 0.]


### 4.2 列空间描述可达到的输出

对 $A\in\mathbb{R}^{m\times n}$，列空间 $\operatorname{Col}(A)\subseteq\mathbb{R}^m$ 是 A 各列的张成。矩阵乘法 $Ax$ 正是在组合这些列，因此

$$Ax=b\text{ 有解}\quad\Longleftrightarrow\quad b\in\operatorname{Col}(A).$$

前面的 columns 三列属于 $\mathbb{R}^3$，其列空间是满足 $b_1=b_2+b_3$ 的平面。它描述哪些右侧 b 可以达到；方程的解集则描述固定 b 后哪些输入 x 可行，两者角色不同。

In [8]:
particular = np.array([1.0, 3.0, 0.0])
right_side = np.array([4.0, 1.0, 3.0])

print(columns @ particular)  # 预期：[4. 1. 3.]，给出列空间内目标的一组系数。
print(columns @ particular - right_side)  # 预期：[0. 0. 0.]。

[4. 1. 3.]
[0. 0. 0.]


### 4.3 零空间描述不改变输出的方向

零空间 $\operatorname{Null}(A)=\{z\in\mathbb{R}^n:Az=0\}$ 是使输出为零的所有输入。它包含零向量；若 Au=Av=0，则 $A(u+v)=0$，且 $A(cu)=0$，因此也是子空间。

对 columns 求齐次方程（右侧为零）：

$$z_1+z_2+2z_3=0,\qquad z_1+z_3=0,\qquad z_2+z_3=0.$$

令 $z_3=t\in\mathbb{R}$，得到 $z=t(-1,-1,1)^{\mathsf T}$。所以零空间是一条过原点的直线。注意列空间位于输出空间 $\mathbb{R}^m$，零空间位于输入空间 $\mathbb{R}^n$；非方阵时它们的向量长度通常不同。

In [9]:
null_direction = np.array([-1.0, -1.0, 1.0])
print(columns @ null_direction)  # 预期：[0. 0. 0.]。

wide = np.array([[1.0, 1.0, 2.0], [1.0, 0.0, 1.0]])
print(wide.shape, null_direction.shape, (wide @ null_direction).shape)  # 三个形状依次为 (2, 3)、(3,)、(2,)。
# 预期：(2, 3) (3,) (2,)；输入空间和输出空间的维数不同。

[0. 0. 0.]
(2, 3) (3,) (2,)


### 4.4 全部解等于特解加零空间

设 $x_p$ 是 Ax=b 的一个特解，$z\in\operatorname{Null}(A)$，则

$$A(x_p+z)=Ax_p+Az=b+0=b.$$

反过来，任意解 x 都满足 $A(x-x_p)=b-b=0$。因此，当方程相容时，全部解恰好是 $x=x_p+z$。

本例 $x_p=(1,3,0)^{\mathsf T}$，于是 $x=(1-t,3-t,t)^{\mathsf T}$，$t\in\mathbb{R}$。右侧非零时，这条解集直线不经过原点，因此不是子空间；它是零空间的平移。

In [10]:
for parameter in (-2.0, 0.0, 1.0):
    solution = particular + parameter * null_direction
    residual = columns @ solution - right_side
    print(solution, residual)  # 三组解依次为 [3, 5, -2]、[1, 3, 0]、[0, 2, 1]；残差均为零向量。
# 三组解依次为 [3, 5, -2]、[1, 3, 0]、[0, 2, 1]；残差均为零。

[ 3.  5. -2.] [0. 0. 0.]
[1. 3. 0.] [0. 0. 0.]
[0. 2. 1.] [0. 0. 0.]


## 5 基、维数与秩

### 5.1 用不冗余的向量描述空间

子空间的一组基（basis）既要张成整个子空间，也要线性无关。上例 v1、v2 已满足这两个条件；加入 v3 虽然仍能张成同一平面，却因相关而不再是一组基。

同一有限维子空间的基可以不同，但包含的向量数相同；这个数称为维数（dimension）。这里列空间维数为 2，虽然每个向量有 3 个坐标。零空间的一组基只需 null_direction，其维数为 1。

消元可以定位列空间的一组基。对 columns 依次做 $R_2\leftarrow R_2-R_1$、$R_3\leftarrow R_3+R_2$（第二步使用更新后的第二行），得到

$$\begin{bmatrix}1&1&2\\0&-1&-1\\0&0&0\end{bmatrix}.$$

主元位于前两列，应选择**原矩阵**的前两列作为列空间的基。行操作通常改变列空间，不能直接拿阶梯形的列代替原来的列。

In [11]:
echelon = columns.copy()
echelon[1] = echelon[1] - echelon[0]
echelon[2] = echelon[2] + echelon[1]
column_basis = columns[:, :2]

print(echelon)  # 主元列是第 1、2 列，第三行全零。
print(column_basis)  # 原矩阵前两列：[[1, 1], [1, 0], [0, 1]]。
print(column_basis @ np.array([1.0, 3.0]))  # 预期：[4. 1. 3.]。

[[ 1.  1.  2.]
 [ 0. -1. -1.]
 [ 0.  0.  0.]]
[[1. 1.]
 [1. 0.]
 [0. 1.]]
[4. 1. 3.]


### 5.2 秩与自由度

矩阵的秩（rank）定义为列空间维数，记为 $\operatorname{rank}(A)$，等于消元得到的主元数。对于 $m\times n$ 矩阵，它不超过 m 和 n 中较小者。

秩—零度定理给出

$$\operatorname{rank}(A)+\dim\operatorname{Null}(A)=n.$$

这里 dim 表示子空间维数，n 是未知数个数。消元把 n 个未知数列分成主元列与自由变量列：前者数量为秩，后者数量为零空间维数，这也解释了等式。

np.linalg.matrix_rank 给出浮点计算下的数值秩。本节的小整数矩阵可以直接用手算主元核对；第 6.2 节再讨论其阈值含义。

In [12]:
rank = np.linalg.matrix_rank(columns)
variable_count = columns.shape[1]
print(rank, variable_count - rank)  # 预期：2 1，对应两个主元和一个自由变量。
print(np.linalg.matrix_rank(column_basis))  # 预期：2，所选两列无关。
print(np.linalg.matrix_rank(np.zeros((2, 3))))  # 预期：0，只有零输出。

2 1
2
0


## 6 判断解的类型与数值边界

### 6.1 先判断相容，再判断唯一

二维时，一条线性方程通常对应一条直线，方程组的解是同时落在两条线上的点。下面把已经出现的三个小系统放在一起，观察“相交、重合、平行而不重合”各对应什么解集。

![正文三个二元系统分别呈现一个交点、重合直线、平行不重合，依次对应唯一解、无穷多解、无解。](image/illustration/03-01-solution-geometry.svg)

图只画直线的一部分；重合直线上的每一个点都是解，不能把屏幕上看到的有限段误当成完整解集。高维系统不能只靠二维直线图判断，仍须检查相容条件与自由变量。

设 $r=\operatorname{rank}(A)$、$r_a=\operatorname{rank}([A\mid b])$，n 为未知数个数。加入 b 后不扩大列空间，恰好意味着 b 在 A 的列空间中；再由自由变量数得到：

| 秩条件 | 解的情况 | 原因 |
| --- | --- | --- |
| $r_a>r$ | 无解 | b 不属于 A 的列空间 |
| $r_a=r=n$ | 唯一解 | 相容，且没有自由变量 |
| $r_a=r<n$ | 无穷多解 | 相容，且有 $n-r$ 个自由变量 |

不能只看“方程数等于未知数数目”，重复方程不增加秩。对于固定 b，满列秩只保证至多一个解，还要检查相容；要求每个 $b\in\mathbb{R}^m$ 都有解则需要满行秩 $r=m$。下面把三组秩与图中的三种关系对照，再考虑浮点误差的边界。

In [13]:
cases = [
    ("unique", a, b),
    ("many", dependent, np.array([3.0, 6.0])),
    ("none", dependent, np.array([3.0, 7.0])),
]
for name, coefficient, right in cases:
    coefficient_rank = np.linalg.matrix_rank(coefficient)
    augmented_rank = np.linalg.matrix_rank(np.column_stack((coefficient, right)))
    # unique 的三列为 2、2、2；many 为 1、1、2；none 为 1、2、2。
    print(name, coefficient_rank, augmented_rank, coefficient.shape[1])
# 依次为 unique 2 2 2、many 1 1 2、none 1 2 2。
# 这里与已手算的消元一致；一般浮点数据的相容判断还要考虑误差尺度。

unique 2 2 2
many 1 1 2
none 1 2 2


### 6.2 数值秩需要阈值

精确数学中，$D=\begin{bmatrix}1&0\\0&\delta\end{bmatrix}$ 在 $\delta\ne0$ 时两列无关，秩为 2；只有 $\delta=0$ 时秩为 1。

matrix_rank 使用奇异值分解（SVD）计算数值秩：统计大于阈值 tol 的奇异值。奇异值是分解产生的一组非负数，这里只用其判断近似退化，不展开分解算法。默认阈值结合最大奇异值、矩阵大小与浮点精度；也可显式指定 tol。

因此，数值秩会把足够微弱的方向视为零。若输入来自测量，阈值还应考虑测量误差与单位尺度；它不是精确线性相关的证明，也不能与检查方程残差的容差混为一谈。

In [14]:
delta = 1e-10
small_direction = np.diag([1.0, delta])

print(np.linalg.matrix_rank(small_direction))  # 本例默认阈值下预期为 2。
print(np.linalg.matrix_rank(small_direction, tol=1e-8))  # 预期为 1。
print(np.linalg.matrix_rank(small_direction, tol=1e-12))  # 预期为 2。
# 矩阵没有改变，改变的是“多小才当作零”的标准；精确数学秩仍是 2。

2
1
2


### 6.3 解出来后仍要代回

对可逆方阵，可以用 np.linalg.solve(A, b) 求解。它要求 A 为方阵且满秩；不满足要求时抛出的异常，不能区分所有无解与多解情形。判断解集仍应使用上面的数学条件。

数值计算应同时检查形状与残差。本例输入和解在 1～3 的尺度，用绝对容差 $10^{-12}$、相对容差 0 判断残差是否接近零。容差选取只用于这个小例子；残差小表示方程满足得较好，不自动证明估计的未知量与真实值接近。

In [15]:
solution = np.linalg.solve(a, b)
residual = a @ solution - b

print(solution)  # 预期：[1. 2.]，与手算一致。
print(residual.shape, residual)  # 预期：(2,) [0. 0.]。
print(np.allclose(residual, np.zeros(2), rtol=0, atol=1e-12))  # 预期：True。

[1. 2.]
(2,) [0. 0.]
True


In [16]:
# 预期 LinAlgError：solve 要求可逆方阵；本系统实际有无穷多解。
np.linalg.solve(dependent, np.array([3.0, 6.0]))

LinAlgError: Singular matrix

## 7 综合应用：增加一条独立观测

用教学设定表示三个实数输入 $x_1,x_2,x_3$ 的两个读数：$y_1=x_1+x_2+2x_3$、$y_2=x_1+x_3$。读数为 $y=(4,1)^{\mathsf T}$ 时，全部输入仍是 $(1-t,3-t,t)^{\mathsf T}$，其中 $t\in\mathbb{R}$；仅凭这两个读数无法区分它们。

现在再测得 $x_3=0.5$。新方程固定自由变量 t，预期唯一解为 $(0.5,2.5,0.5)^{\mathsf T}$。若只是把已有第一条观测再重复一次，就不会获得独立信息，秩也不会增加。

下面先比较两种新增方程的秩，再求解独立观测形成的方阵；最后核对全部三个方程的残差。

In [17]:
original = np.array([[1.0, 1.0, 2.0], [1.0, 0.0, 1.0]])
repeated = np.array([[1.0, 1.0, 2.0], [1.0, 0.0, 1.0], [1.0, 1.0, 2.0]])
extended = np.array([[1.0, 1.0, 2.0], [1.0, 0.0, 1.0], [0.0, 0.0, 1.0]])
observations = np.array([4.0, 1.0, 0.5])

print(np.linalg.matrix_rank(original), np.linalg.matrix_rank(repeated))  # 两个秩都为 2；复制已有行没有增加独立约束。
# 预期：2 2；重复一条方程没有消除自由度。
print(np.linalg.matrix_rank(extended))  # 预期：3；新增观测消除剩余自由度。
recovered = np.linalg.solve(extended, observations)
print(recovered)  # 预期：[0.5 2.5 0.5]。
print(extended @ recovered - observations)  # 预期：[0. 0. 0.]。

2 2
3
[0.5 2.5 0.5]
[0. 0. 0.]


## 本章小结

（1）消元保持解集。先排除矛盾行，再用主元和自由变量描述全部解，不只寻找一个候选解。

（2）张成收集所有线性组合；线性无关排除冗余；基同时满足张成与无关，基向量数量给出维数。

（3）列空间描述可达到的输出，零空间描述产生零输出的输入。有解时，全部解是特解加零空间。

（4）秩连接主元数、列空间维数和解的自由度。浮点数值秩依赖阈值，残差检查也需要结合尺度。

理解自查：为什么三个方程、三个未知数仍可能没有唯一解？为什么找列空间的基要回到原矩阵取列？增加一次测量为什么不一定增加信息？

## 练习

（1）手算消元并代回

对下面的二元系统写出增广矩阵，用行操作消元后求解，再与 solve 的结果比较。

可验证标准：保留至少一次明确的行操作，两个未知数都代回原系统，残差各分量的绝对值不超过 $10^{-12}$。

In [18]:
exercise_a = np.array([[2.0, 1.0], [1.0, -1.0]])
exercise_b = np.array([5.0, 1.0])
# 在这里写消元步骤、候选解与残差检查。

（2）改变右侧条件

保持 dependent 不变，右侧改为 $(k,6)^{\mathsf T}$，其中 $k\in\mathbb{R}$。先推导哪个 k 使系统相容，再分别选择一个相容值和一个不相容值核对消元与秩。

可验证标准：相容情形写出含一个自由参数的全部解；不相容情形指出消元后的矛盾行。不能只根据 solve 是否报错判断。

In [19]:
exercise_dependent = np.array([[1.0, 2.0], [2.0, 4.0]])
# 在这里选择 k 并构造右侧；分别说明两种情况的数学依据。

（3）分别求列空间与零空间的基

对下面的 $2\times3$ 矩阵，手算消元，选择原矩阵的主元列形成列空间的基；再求齐次方程的参数形式，写出零空间的一组基。

可验证标准：明确两种基向量各有几个分量；核对矩阵乘每个零空间基向量为零，并检查秩与零空间维数之和为 3。

In [20]:
exercise_wide = np.array([[1.0, 2.0, 3.0], [0.0, 1.0, 1.0]])
# 在这里写阶梯形、两种基及核对计算。

（4）给解集增加非负约束

综合应用中若三个输入还要求非负，在增加第三条观测之前，参数 t 可以取哪些值？加入 $x_3=0.5$ 后是否仍有解？若测得 $x_3=2$ 呢？

可验证标准：先从三个分量分别写出不等式，求共同区间；区分“线性方程有解”与“同时满足非负约束”。用选定的边界值和区间外值核对，但区间结论要来自不等式推导。

In [21]:
# 输入形式为 [1-t, 3-t, t]；在这里选择 t，打印输入与非负判断。

## 练习提示与解析

以下对应练习（4）。先独立作答，卡住时依次查看提示，完成后再对照解析。

提示 1：把非负要求分别施加到三个分量，而不是只检查自由参数。

提示 2：解集中的每个候选都满足原方程；新增不等式会缩小允许的参数区间。

### 练习（4）参考解析

由 $(1-t,3-t,t)^T\ge0$ 分别得到 $t\le1$、$t\le3$、$t\ge0$，共同区间为 $[0,1]$。端点给出 $(1,3,0)^T$ 与 $(0,2,1)^T$，都合法。

增加 $x_3=0.5$ 即固定 $t=0.5$，得到 $(0.5,2.5,0.5)^T$。增加 $x_3=2$ 时，线性方程仍有解 $(-1,1,2)^T$，但第一分量为负，因此没有同时满足非负约束的解。

## 参考与引用来源

本章新增示意图由 CMYK Labs 原创，依据下表对应概念与本章教学输入绘制；示意图不作为实际运行截图或数学证明。

以下页面于 2026-09-21 核查。示例数值与教学观测为本章编写；矩阵运算用于核对具体输入，不替代通用结论的证明。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Georgia Tech：Interactive Linear Algebra | [§1.1 Systems of Linear Equations](https://textbooks.math.gatech.edu/ila/systems-of-eqns.html)：解集、相容与不相容；[§1.2 Row Reduction](https://textbooks.math.gatech.edu/ila/row-reduction.html)：§1.2.1–1.2.3 行操作、阶梯形、主元与矛盾行；[§1.3 Parametric Form](https://textbooks.math.gatech.edu/ila/parametric-form.html)：自由变量及三种解数量；[§2.2 Vector Equations and Spans](https://textbooks.math.gatech.edu/ila/spans.html)：§2.2.2 张成与相容；[§2.4 Solution Sets](https://textbooks.math.gatech.edu/ila/solution-sets.html)：§2.4.1 齐次方程，§2.4.2 特解加齐次解，§2.4.3 解集与列空间区别；[§2.5 Linear Independence](https://textbooks.math.gatech.edu/ila/linear-independence.html)：§2.5.1 定义、零向量，§2.5.2 冗余向量；[§2.6 Subspaces](https://textbooks.math.gatech.edu/ila/subspaces.html)：§2.6.1 子空间条件，§2.6.2 列空间与零空间；[§2.7 Basis and Dimension](https://textbooks.math.gatech.edu/ila/dimension.html)：§2.7.1 基与维数，§2.7.2 原矩阵主元列与零空间基；[§2.9 The Rank Theorem](https://textbooks.math.gatech.edu/ila/rank-thm.html)：秩、零度、主元与自由变量。第 6.1 节的秩判据由列空间相容条件与自由变量数整理。 |
| NumPy 2.5 官方文档 | [linalg.solve](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html)：方阵满秩条件、输入输出形状与 LinAlgError；[linalg.matrix_rank](https://numpy.org/doc/stable/reference/generated/numpy.linalg.matrix_rank.html)：tol、默认阈值、Notes 中舍入及测量误差的区别；[column_stack](https://numpy.org/doc/stable/reference/generated/numpy.column_stack.html)：按列组合一维与二维数组；[diag](https://numpy.org/doc/stable/reference/generated/numpy.diag.html)：从一维数组构造对角矩阵；[allclose](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html)：绝对、相对容差与广播边界。 |
| SciPy 官方文档 | [linalg.svd](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.svd.html)：开篇的奇异值为实数、非负数说明；本章仅用来解释数值秩的输入量，不调用该函数。 |
| Fundamentals of Numerical Computation 作者教材 | [§2.8 Conditioning of linear systems](https://fncbook.com/condition-number/)：Residual and backward error，小残差与解误差的区别。本章残差采用 A @ x - b，与该页的符号相反，但绝对大小相同。 |